# ReadyNow! — FEMA Emergency Preparedness Agent

A multi-agent system built with Google ADK that helps people get
real-time updates during disasters, find evacuation routes, and
stay informed about emergency safety procedures.

**Capabilities:**
- Real-time weather conditions and NWS alerts
- Internet search for disaster news and updates
- Evacuation route planning via Google Routes API (safe zone: Denver, CO)
- Emergency preparedness Q&A
- Synthetic disaster simulation for testing

**Architecture:**
- Root agent with LLM-based routing
- Parallel research phase (weather + news)
- Sequential validation and refinement pipeline
- Callback-based interaction logging and input validation

## Architecture Diagram

```
                        ┌──────────────────────┐
                        │   readynow_root      │
                        │   (Agent — router)    │
                        │                      │
                        │  Callbacks:           │
                        │  • log_interaction    │
                        │  • validate_input     │
                        └──┬───────┬────────┬──┘
                           │       │        │
              ┌────────────┘       │        └─────────────┐
              ▼                    ▼                      ▼
   ┌──────────────────┐  ┌─────────────────┐  ┌─────────────────────────────┐
   │  greeter_agent   │  │  safety_agent   │  │  emergency_pipeline         │
   │  (LlmAgent)      │  │  (LlmAgent)     │  │  (SequentialAgent)          │
   │                  │  │                 │  │                             │
   │  Greetings &     │  │  General Q&A    │  │  ┌────────────────────────┐ │
   │  casual chat     │  │  about safety   │  │  │ triage_agent           │ │
   └──────────────────┘  │  & preparedness │  │  │ → state["triage"]      │ │
                         └─────────────────┘  │  └────────────────────────┘ │
                                              │             │               │
                                              │             ▼               │
                                              │  ┌────────────────────────┐ │
                                              │  │ research_phase         │ │
                                              │  │ (ParallelAgent)        │ │
                                              │  │  ├─ weather_agent      │ │
                                              │  │  │  → state["weather"] │ │
                                              │  │  └─ news_agent        │ │
                                              │  │     → state["news"]   │ │
                                              │  └────────────────────────┘ │
                                              │             │               │
                                              │             ▼               │
                                              │  ┌────────────────────────┐ │
                                              │  │ action_agent           │ │
                                              │  │ (evacuation routes +   │ │
                                              │  │  safety recs)          │ │
                                              │  │ → state["draft"]       │ │
                                              │  └────────────────────────┘ │
                                              │             │               │
                                              │             ▼               │
                                              │  ┌────────────────────────┐ │
                                              │  │ validate_agent         │ │
                                              │  │ → state["validation"]  │ │
                                              │  └────────────────────────┘ │
                                              │             │               │
                                              │             ▼               │
                                              │  ┌────────────────────────┐ │
                                              │  │ refine_agent           │ │
                                              │  │ → final output         │ │
                                              │  └────────────────────────┘ │
                                              └─────────────────────────────┘
```

## 1. Setup & Dependencies

In [ ]:
!pip install google-cloud-aiplatform[agent_engines,adk] -q
!pip install google-genai -q

## 2. Configuration

In [ ]:
import os
from typing import Optional

import google.auth
import vertexai
from google.genai import types
from google import genai

from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LlmAgent
from google.adk.tools.tool_context import ToolContext
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from vertexai import agent_engines
from vertexai.agent_engines import AdkApp

project = google.auth.default()[1]
LOCATION = "us-central1"
GEMINI_MODEL = "gemini-2.0-flash"
STAGING_BUCKET = f"gs://{project}-agent-engine-staging"
SAFE_ZONE = "Denver, CO, USA"

# ──── Replace with your Google Maps Platform API key ────
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY"

os.environ["GOOGLE_MAPS_API_KEY"] = GOOGLE_MAPS_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = project
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(
    project=project,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Project:        {project}")
print(f"Location:       {LOCATION}")
print(f"Model:          {GEMINI_MODEL}")
print(f"Safe Zone:      {SAFE_ZONE}")
print(f"Staging bucket: {STAGING_BUCKET}")

In [ ]:
!gsutil ls $STAGING_BUCKET 2>/dev/null || gsutil mb -l $LOCATION $STAGING_BUCKET
print(f"Staging bucket ready: {STAGING_BUCKET}")

## 3. Tool Functions

All tools are **self-contained** (imports inside the function body)
so they serialize correctly for Agent Engine deployment.

| Tool | Purpose |
|---|---|
| `get_weather_and_alerts` | NWS forecast + active alerts for a US location |
| `search_the_web` | Google Search via Gemini grounding |
| `compute_evacuation_route` | Google Routes API → Denver, CO safe zone |
| `simulate_disaster` | Inject synthetic disaster alerts for testing |

In [ ]:
def get_weather_and_alerts(location: str) -> str:
    """Get current weather forecast and active NWS alerts for a US location.

    Geocodes the location, then queries the National Weather Service for
    the current forecast and any active weather alerts.

    Args:
        location: A US city or address (e.g., 'Miami, FL').

    Returns:
        A formatted string with weather conditions and any active alerts.
    """
    import requests
    import os
    import time

    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")
    headers = {"User-Agent": "ReadyNow-FEMA-Agent/1.0"}

    # Geocode the location
    try:
        time.sleep(1)
        geo_url = "https://maps.googleapis.com/maps/api/geocode/json"
        geo_resp = requests.get(geo_url, params={"address": location, "key": api_key})
        geo_resp.raise_for_status()
        geo_data = geo_resp.json()

        if not geo_data.get("results"):
            return f"Could not geocode location: {location}"

        lat = geo_data["results"][0]["geometry"]["location"]["lat"]
        lng = geo_data["results"][0]["geometry"]["location"]["lng"]
    except Exception as e:
        return f"Geocoding failed for '{location}': {e}"

    # NWS forecast
    forecast_text = "Forecast unavailable."
    try:
        points_resp = requests.get(
            f"https://api.weather.gov/points/{lat},{lng}", headers=headers
        )
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        forecast_resp = requests.get(forecast_url, headers=headers)
        forecast_resp.raise_for_status()
        period = forecast_resp.json()["properties"]["periods"][0]
        forecast_text = (
            f"{period['name']}: {period['detailedForecast']}\n"
            f"Temperature: {period['temperature']}°{period['temperatureUnit']}"
        )
    except Exception as e:
        forecast_text = f"Forecast error: {e}"

    # NWS active alerts
    alert_text = "No active alerts."
    try:
        alerts_resp = requests.get(
            f"https://api.weather.gov/alerts/active?point={lat},{lng}",
            headers=headers,
        )
        if alerts_resp.ok:
            features = alerts_resp.json().get("features", [])
            if features:
                lines = []
                for f in features[:5]:
                    props = f["properties"]
                    lines.append(
                        f"- {props.get('event', 'Alert')}: "
                        f"{props.get('headline', 'N/A')}"
                    )
                alert_text = "\n".join(lines)
    except Exception:
        pass

    return (
        f"Weather for {location}:\n"
        f"{forecast_text}\n\n"
        f"Active Alerts:\n{alert_text}"
    )


print("get_weather_and_alerts defined.")

In [ ]:
def search_the_web(query: str) -> str:
    """Search the web for information using Google Search.

    Use this tool to find up-to-date news, disaster reports,
    emergency declarations, and official advisories.

    Args:
        query: The search query to look up.

    Returns:
        A text summary of the search results.
    """
    from google import genai as _genai
    from google.genai import types as _types
    import google.auth as _auth

    _project = _auth.default()[1]
    client = _genai.Client(vertexai=True, project=_project, location="us-central1")
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=query,
        config=_types.GenerateContentConfig(
            tools=[_types.Tool(google_search=_types.GoogleSearch())],
        ),
    )
    return response.text


print("search_the_web defined.")

In [ ]:
def compute_evacuation_route(origin: str) -> str:
    """Compute a driving evacuation route from origin to the designated safe zone (Denver, CO).

    Uses the Google Routes API to calculate distance, duration, and
    route description for evacuation planning.

    Args:
        origin: Starting location for evacuation (e.g., 'Miami, FL').

    Returns:
        A formatted evacuation route summary with distance, duration,
        and route description.
    """
    import requests
    import os

    api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")
    safe_zone = "Denver, CO, USA"

    url = "https://routes.googleapis.com/directions/v2:computeRoutes"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": (
            "routes.duration,routes.distanceMeters,"
            "routes.description,routes.warnings"
        ),
    }
    body = {
        "origin": {"address": origin},
        "destination": {"address": safe_zone},
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE",
    }

    try:
        resp = requests.post(url, json=body, headers=headers)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"Route calculation failed: {e}"

    if not data.get("routes"):
        return f"No route found from {origin} to {safe_zone}."

    route = data["routes"][0]
    distance_m = route.get("distanceMeters", 0)
    distance_miles = distance_m * 0.000621371
    duration_sec = int(route.get("duration", "0s").replace("s", ""))
    duration_hrs = duration_sec / 3600
    description = route.get("description", "Route details unavailable")
    warnings = route.get("warnings", [])

    result = (
        f"EVACUATION ROUTE: {origin} → {safe_zone}\n"
        f"Route: {description}\n"
        f"Distance: {distance_miles:.0f} miles\n"
        f"Estimated Duration: {duration_hrs:.1f} hours\n"
    )

    if warnings:
        result += f"Warnings: {'; '.join(warnings)}\n"

    result += (
        "\nNote: Duration based on current traffic. Actual evacuation "
        "time may increase due to emergency traffic volume."
    )
    return result


print("compute_evacuation_route defined.")

In [ ]:
def simulate_disaster(disaster_type: str, location: str, severity: str) -> str:
    """Simulate a disaster alert for testing the ReadyNow! system.

    This is a testing backdoor that generates synthetic emergency
    alerts. Use this when the user asks to simulate or test a
    disaster scenario.

    Args:
        disaster_type: Type of disaster — hurricane, earthquake, tornado,
            wildfire, tsunami, volcanic_eruption, flood, or godzilla.
        location: City or region (e.g., 'Miami, FL').
        severity: Alert level — advisory, watch, warning, or emergency.

    Returns:
        A formatted simulated emergency alert bulletin.
    """
    import datetime

    severity_map = {
        "advisory": "Conditions are possible. Stay informed and monitor updates.",
        "watch": "Conditions are favorable for development. Prepare to act.",
        "warning": "Dangerous conditions imminent or occurring. Take action NOW.",
        "emergency": "EXTREME DANGER. Life-threatening situation. Evacuate immediately.",
    }

    detail_map = {
        "hurricane": (
            "Sustained winds exceeding 74 mph with dangerous storm surge. "
            "Coastal flooding expected. Board windows, secure loose objects, "
            "and evacuate if ordered."
        ),
        "earthquake": (
            "Significant seismic activity detected. Aftershocks likely. "
            "Drop, cover, and hold on. Check for gas leaks and structural damage."
        ),
        "tornado": (
            "Rotating thunderstorm with confirmed funnel cloud. "
            "Seek interior shelter on lowest floor immediately. Avoid windows."
        ),
        "wildfire": (
            "Rapidly spreading wildfire with poor containment. "
            "Dangerous air quality. Prepare for possible evacuation orders."
        ),
        "tsunami": (
            "Large ocean wave activity detected following seismic event. "
            "Move to high ground immediately. Stay away from coast."
        ),
        "volcanic_eruption": (
            "Active volcanic eruption with ash fall and possible lava flow. "
            "Evacuate danger zones. Wear masks to protect from ash inhalation."
        ),
        "flood": (
            "Water levels rising rapidly due to heavy rainfall. "
            "Move to higher ground. Never drive through flooded roads."
        ),
        "godzilla": (
            "Giant radioactive creature detected offshore, moving inland. "
            "Military response initiated. Evacuate coastal areas immediately. "
            "Avoid all major bridges and tunnels."
        ),
    }

    sev_desc = severity_map.get(
        severity.lower(), "Unknown severity level. Stay alert."
    )
    details = detail_map.get(
        disaster_type.lower(),
        f"{disaster_type} event reported. Follow local emergency guidance.",
    )
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M UTC")

    return (
        f"{'=' * 50}\n"
        f"  SIMULATED EMERGENCY ALERT\n"
        f"  (THIS IS A TEST — NOT A REAL ALERT)\n"
        f"{'=' * 50}\n"
        f"Type:     {disaster_type.upper()}\n"
        f"Location: {location}\n"
        f"Severity: {severity.upper()}\n"
        f"Time:     {now}\n\n"
        f"Status:   {sev_desc}\n\n"
        f"Details:  {details}\n\n"
        f"RECOMMENDED ACTIONS:\n"
        f"1. Monitor official channels (NOAA, local EMA) for updates\n"
        f"2. Follow local evacuation orders if issued\n"
        f"3. Prepare emergency kit (water, food, medications, documents)\n"
        f"4. Contact family members and establish a meeting point\n"
        f"5. If severity is WARNING or EMERGENCY, evacuate to the\n"
        f"   designated safe zone (Denver, CO)\n"
        f"{'=' * 50}"
    )


print("simulate_disaster defined.")

## 4. Callback Functions

| Callback | Hook | Purpose |
|---|---|---|
| `log_before_model` | `before_model_callback` | Logs every user-to-agent prompt |
| `log_after_model` | `after_model_callback` | Logs every agent response |
| `validate_input` | `before_model_callback` | Blocks off-topic requests at the root agent |

In [ ]:
def _get_latest_user_text(llm_request: LlmRequest) -> Optional[str]:
    """Extract the most recent user text, skipping tool-result turns."""
    if not llm_request.contents:
        return None
    for content in reversed(llm_request.contents):
        if content.role != "user" or not content.parts:
            continue
        if any(getattr(p, "function_response", None) for p in content.parts):
            continue
        for part in content.parts:
            if getattr(part, "text", None):
                return part.text
    return None


def log_before_model(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Log user prompts before they reach the model."""
    user_text = _get_latest_user_text(llm_request)
    if user_text:
        print(
            f"  [LOG] User → {callback_context.agent_name}: "
            f"{user_text[:120]}{'...' if len(user_text) > 120 else ''}"
        )
    return None


def log_after_model(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Log model responses after generation."""
    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if getattr(part, "text", None):
                preview = part.text[:150]
                print(
                    f"  [LOG] {callback_context.agent_name} → "
                    f"{preview}{'...' if len(part.text) > 150 else ''}"
                )
            elif getattr(part, "function_call", None):
                print(
                    f"  [LOG] {callback_context.agent_name} → "
                    f"tool: {part.function_call.name}()"
                )
    return None


def validate_input(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Reject off-topic requests at the root agent level.

    Only fires for the root agent. Checks whether the user's message
    is related to emergency preparedness, weather, disasters, or safety.
    Short messages (greetings) are allowed through.
    """
    if callback_context.agent_name != "readynow_root":
        return None

    user_text = _get_latest_user_text(llm_request)
    if not user_text:
        return None

    # Allow short messages (likely greetings)
    if len(user_text.split()) <= 6:
        return None

    emergency_keywords = [
        "weather", "storm", "hurricane", "tornado", "earthquake", "flood",
        "fire", "wildfire", "evacuate", "evacuation", "emergency", "disaster",
        "safety", "shelter", "alert", "warning", "prepare", "supplies",
        "tsunami", "volcano", "godzilla", "simulate", "route", "directions",
        "forecast", "rain", "snow", "wind", "temperature", "hazard",
        "fema", "readynow", "help", "what should", "how do i", "where",
        "safe zone", "denver", "danger", "threat", "damage", "rescue",
        "survival", "kit", "plan", "family", "water", "food", "power",
        "outage", "road", "closure", "news", "update",
    ]

    text_lower = user_text.lower()
    if any(kw in text_lower for kw in emergency_keywords):
        return None

    print(f"  [BLOCKED] Off-topic: {user_text[:100]}")
    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[types.Part(text=(
                "I'm ReadyNow!, FEMA's emergency preparedness assistant. "
                "I can help with:\n"
                "- Current weather conditions and alerts\n"
                "- Disaster information and safety guidance\n"
                "- Evacuation routes to safe zones\n"
                "- Emergency preparedness tips and planning\n\n"
                "Please ask me something related to emergency preparedness!"
            ))],
        )
    )


print("Callbacks defined.")

## 5. Agent Definitions

### Agent Hierarchy

- **readynow_root** — Routes to greeter, safety Q&A, or emergency pipeline
- **greeter_agent** — Handles greetings and casual chat
- **safety_agent** — Answers general emergency preparedness questions
- **emergency_pipeline** (SequentialAgent):
  1. **triage_agent** — Classifies the request
  2. **research_phase** (ParallelAgent):
     - **weather_agent** — NWS forecast + alerts + disaster simulation
     - **news_agent** — Web search for disaster news
  3. **action_agent** — Synthesizes research + computes evacuation routes
  4. **validate_agent** — Checks response quality
  5. **refine_agent** — Polishes final output

In [ ]:
# ── Greeter Agent ──

greeter_agent = LlmAgent(
    name="greeter_agent",
    model=GEMINI_MODEL,
    description=(
        "Handles greetings and casual conversation. Use for hello, hi, "
        "thanks, goodbye, or small talk."
    ),
    instruction="""
You are a friendly ReadyNow! assistant. Respond warmly to the user's
greeting. Briefly mention that you can help with weather updates,
disaster alerts, evacuation routes, and safety information.
Keep it to 2-3 sentences.
""",
)

# ── Safety Q&A Agent ──

safety_agent = LlmAgent(
    name="safety_agent",
    model=GEMINI_MODEL,
    description=(
        "Answers general emergency preparedness questions — what to pack "
        "in an emergency kit, how to prepare for specific disasters, "
        "family emergency plans, shelter-in-place procedures, etc. "
        "Use this when the user asks general safety or preparedness "
        "questions that do NOT require real-time weather or news data."
    ),
    instruction="""
You are a FEMA emergency preparedness expert. Answer the user's
question with clear, actionable safety guidance.

Structure your response with:
1. Direct answer to their question
2. Step-by-step recommendations if applicable
3. Key supplies or resources they should have
4. Where to find more information (ready.gov, local EMA)

Be thorough but concise. Use simple language accessible to all
reading levels. Prioritize life-safety information first.
""",
)

print("greeter_agent and safety_agent defined.")

In [ ]:
# ── Pipeline Step 1: Triage Agent ──

triage_agent = LlmAgent(
    name="triage_agent",
    model=GEMINI_MODEL,
    description="Classifies the user's emergency request.",
    instruction="""
You are the triage agent for ReadyNow!. Analyze the user's request
and produce a brief classification to guide downstream agents.

Output a structured triage in this format:
- REQUEST_TYPE: [WEATHER | DISASTER_SIM | ACTIVE_DISASTER | EVACUATION | MIXED]
- LOCATION: [extracted location or "not specified"]
- DISASTER_TYPE: [type if applicable, or "none"]
- SEVERITY: [advisory | watch | warning | emergency | unknown]
- SUMMARY: [one-sentence summary of what the user needs]

Be concise. This output guides other agents, not the user.
""",
    output_key="triage",
)

# ── Pipeline Step 2: Research Phase (Parallel) ──

weather_agent = LlmAgent(
    name="weather_agent",
    model=GEMINI_MODEL,
    description="Retrieves weather data and processes disaster simulations.",
    instruction="""
You are the weather data agent for ReadyNow!.

Triage: {triage}

Based on the triage:
1. If a LOCATION is specified, use get_weather_and_alerts to get
   real weather data and any active NWS alerts.
2. If REQUEST_TYPE is DISASTER_SIM, also use simulate_disaster
   to generate a synthetic alert with the specified type, location,
   and severity.
3. Report ALL findings — both real weather and any simulated alerts.

Present the data factually. Do not add recommendations (the action
agent handles that).
""",
    tools=[get_weather_and_alerts, simulate_disaster],
    output_key="weather_data",
)

news_agent = LlmAgent(
    name="news_agent",
    model=GEMINI_MODEL,
    description="Searches for relevant disaster news and updates.",
    instruction="""
You are the news research agent for ReadyNow!.

Triage: {triage}

Search for the latest news related to the user's situation:
- Current disaster reports or emergency declarations
- Official government advisories
- Road closures or infrastructure impacts
- Shelter locations or resource distribution points

Use the search_the_web tool. Focus on reliable sources (NOAA,
FEMA, local emergency management, major news outlets).

If this is a DISASTER_SIM (simulation), search for general
information about that type of disaster to add realism.

Report only factual findings.
""",
    tools=[search_the_web],
    output_key="news_data",
)

research_phase = ParallelAgent(
    name="research_phase",
    sub_agents=[weather_agent, news_agent],
    description="Gathers weather and news data in parallel.",
)

print("triage_agent, weather_agent, news_agent, research_phase defined.")

In [ ]:
# ── Pipeline Step 3: Action Agent ──

action_agent = LlmAgent(
    name="action_agent",
    model=GEMINI_MODEL,
    description="Synthesizes research and provides actionable guidance.",
    instruction="""
You are the action and response agent for ReadyNow!.

## Triage
{triage}

## Weather Data
{weather_data}

## News Updates
{news_data}

Based on all available information, create a comprehensive
emergency response for the user:

1. **SITUATION SUMMARY**: What is happening and where
2. **CURRENT CONDITIONS**: Weather and any active alerts
3. **EVACUATION ROUTE**: If severity is WARNING or EMERGENCY,
   or if the user asked about evacuation, use the
   compute_evacuation_route tool to calculate a route from
   the affected location to the safe zone (Denver, CO).
   Include the route details in your response.
4. **SAFETY RECOMMENDATIONS**: Specific actions to take now
5. **RESOURCES**: Emergency contacts and information sources

For simulated disasters, clearly mark the response as a
simulation/drill while still providing realistic guidance.

Prioritize life-safety information. Be clear and direct.
""",
    tools=[compute_evacuation_route],
    output_key="draft_response",
)

# ── Pipeline Step 4: Validate Agent ──

validate_agent = LlmAgent(
    name="validate_agent",
    model=GEMINI_MODEL,
    description="Validates the draft response for quality and accuracy.",
    instruction="""
You are the quality validator for FEMA's ReadyNow! system.
Review the following draft response:

## Draft Response
{draft_response}

Check that the response:
1. Is factually accurate and not misleading
2. Includes specific, actionable safety guidance
3. Is written in clear, simple language (8th grade reading level)
4. Clearly distinguishes simulated alerts from real ones
5. Does not cause unnecessary panic
6. Includes evacuation information if the situation warrants it

Provide specific feedback. If the response is acceptable, say
"APPROVED" and note any minor suggestions. If critical issues
exist, list them clearly.
""",
    output_key="validation",
)

# ── Pipeline Step 5: Refine Agent ──

refine_agent = LlmAgent(
    name="refine_agent",
    model=GEMINI_MODEL,
    description="Produces the final polished response.",
    instruction="""
You are the final editor for FEMA's ReadyNow! system.
Take the draft response and validation feedback and produce
a polished final response.

## Draft Response
{draft_response}

## Validation Feedback
{validation}

Produce a final response that:
- Addresses all validation feedback
- Is well-structured with clear sections and headers
- Uses simple, direct language anyone can understand
- Leads with the most critical safety information
- Includes a clear "WHAT TO DO NOW" section
- Keeps total length under 400 words

Output ONLY the final polished response for the user.
""",
    output_key="final_response",
)

print("action_agent, validate_agent, refine_agent defined.")

In [ ]:
# ── Assemble Pipeline & Root Agent ──

emergency_pipeline = SequentialAgent(
    name="emergency_pipeline",
    description=(
        "Handles all emergency-related queries: weather lookups, "
        "disaster alerts (real or simulated), evacuation route planning, "
        "and situational analysis. Use this for any request involving "
        "real-time weather, disaster scenarios, or evacuation needs."
    ),
    sub_agents=[
        triage_agent,
        research_phase,
        action_agent,
        validate_agent,
        refine_agent,
    ],
)

root_agent = Agent(
    name="readynow_root",
    model=GEMINI_MODEL,
    description="ReadyNow! FEMA Emergency Preparedness Agent — main coordinator.",
    instruction="""
You are ReadyNow!, FEMA's Emergency Preparedness Chat Agent.
Your mission is to help people get real-time updates during disasters,
find evacuation routes, and stay safe.

Route each user request to the appropriate specialist:

- **greeter_agent**: For greetings, hellos, thanks, goodbyes,
  and casual conversation.

- **safety_agent**: For general emergency preparedness questions
  that do NOT need real-time data — e.g., "What should I pack in
  an emergency kit?", "How do I prepare for a hurricane?",
  "What is shelter-in-place?"

- **emergency_pipeline**: For anything requiring real-time data:
  weather queries, disaster alerts, simulated disasters, evacuation
  route requests, or situational updates. This includes:
  - "What's the weather in Miami?"
  - "Simulate a Category 5 hurricane in Houston"
  - "I need an evacuation route from New Orleans"
  - "Are there any active alerts in California?"

ALWAYS delegate. Never answer directly. If unclear, ask a brief
clarifying question.
""",
    sub_agents=[greeter_agent, safety_agent, emergency_pipeline],
    before_model_callback=[log_before_model, validate_input],
    after_model_callback=log_after_model,
)

print("emergency_pipeline and root_agent assembled.")
print("\nAgent hierarchy:")
print("  readynow_root (Agent — router)")
print("  ├── greeter_agent (LlmAgent)")
print("  ├── safety_agent (LlmAgent)")
print("  └── emergency_pipeline (SequentialAgent)")
print("      ├── triage_agent")
print("      ├── research_phase (ParallelAgent)")
print("      │   ├── weather_agent")
print("      │   └── news_agent")
print("      ├── action_agent")
print("      ├── validate_agent")
print("      └── refine_agent")

## 6. Local Testing with AdkApp

Wrap the agent in `AdkApp` and run test scenarios locally.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*is deprecated.*")

app = AdkApp(agent=root_agent)


def run_test(message: str, user_id: str = "test_user"):
    """Run a local test query and print results."""
    session = app.create_session(user_id=user_id)
    print(f"\n{'=' * 70}")
    print(f"User: {message}")
    print(f"{'=' * 70}")

    for event in app.stream_query(
        user_id=user_id,
        session_id=session.id,
        message=message,
    ):
        author = event.get("author", "agent")
        content = event.get("content", {})
        for part in content.get("parts", []):
            if "text" in part:
                print(f"  [{author}] {part['text'][:300]}")
            elif "function_call" in part:
                fn = part["function_call"]["name"]
                print(f"  [{author}] → {fn}()")


print("AdkApp and test helper ready.")

### 6a. Greeting Test

Should route to `greeter_agent`.

In [ ]:
run_test("Hello! What can you help me with?")

### 6b. Safety Q&A Test

Should route to `safety_agent` (no real-time data needed).

In [ ]:
run_test("What should I have in my emergency preparedness kit?")

### 6c. Weather Query Test

Should route to `emergency_pipeline` → weather + news + action.

In [ ]:
run_test("What are the current weather conditions and any active alerts in Miami, FL?")

### 6d. Disaster Simulation + Evacuation Test

Should simulate a hurricane, fetch real weather, compute evacuation
route to Denver, and provide safety guidance.

In [ ]:
run_test(
    "Simulate a Category 5 hurricane emergency in Houston, TX. "
    "I need an evacuation route to safety."
)

### 6e. Off-Topic Rejection Test

Should be blocked by the `validate_input` callback.

In [ ]:
run_test("Write me a sonnet about the beauty of autumn leaves falling gently.")

### 6f. Godzilla Attack Simulation

Stress-test with the most exotic disaster type.

In [ ]:
run_test(
    "EMERGENCY: Simulate a Godzilla attack on San Francisco, CA! "
    "Severity: emergency. I need to evacuate NOW!"
)

## 7. Deploy to Agent Engine

Deploy the agent to Vertex AI Agent Engine for production use.
This packages the agent and all tools, uploads to the staging
bucket, and creates a managed endpoint.

In [ ]:
REQUIREMENTS = [
    "google-cloud-aiplatform[agent_engines,adk]",
    "google-genai",
    "cloudpickle",
    "requests",
]

remote_agent = agent_engines.create(
    agent_engine=app,
    requirements=REQUIREMENTS,
    display_name="readynow-fema-agent",
)

print(f"\nDeployed successfully!")
print(f"Resource name: {remote_agent.resource_name}")

## 8. Remote Testing

Test the deployed agent running on Agent Engine.

### 8a. Remote — Greeting

In [ ]:
remote_session = remote_agent.create_session(user_id="remote_user")
print(f"Remote session: {remote_session}\n")

for event in remote_agent.stream_query(
    user_id="remote_user",
    session_id=remote_session["id"],
    message="Hello! What can ReadyNow! do for me?",
):
    author = event.get("author", "agent") if isinstance(event, dict) else "agent"
    content = event.get("content", {}) if isinstance(event, dict) else {}
    for part in content.get("parts", []):
        if "text" in part:
            print(f"  [{author}] {part['text'][:300]}")

### 8b. Remote — Weather Query

In [ ]:
remote_session2 = remote_agent.create_session(user_id="remote_user")

for event in remote_agent.stream_query(
    user_id="remote_user",
    session_id=remote_session2["id"],
    message="What is the weather in Chicago, IL? Are there any active alerts?",
):
    author = event.get("author", "agent") if isinstance(event, dict) else "agent"
    content = event.get("content", {}) if isinstance(event, dict) else {}
    for part in content.get("parts", []):
        if "text" in part:
            print(f"  [{author}] {part['text'][:300]}")

### 8c. Remote — Disaster Simulation + Evacuation

In [ ]:
remote_session3 = remote_agent.create_session(user_id="remote_user")

for event in remote_agent.stream_query(
    user_id="remote_user",
    session_id=remote_session3["id"],
    message=(
        "Simulate a tornado warning in Oklahoma City, OK. "
        "What should I do and can you give me an evacuation route?"
    ),
):
    author = event.get("author", "agent") if isinstance(event, dict) else "agent"
    content = event.get("content", {}) if isinstance(event, dict) else {}
    for part in content.get("parts", []):
        if "text" in part:
            print(f"  [{author}] {part['text'][:300]}")

### 8d. Remote — Off-Topic Rejection

In [ ]:
remote_session4 = remote_agent.create_session(user_id="remote_user")

for event in remote_agent.stream_query(
    user_id="remote_user",
    session_id=remote_session4["id"],
    message="Can you help me write a Python script to sort a list of numbers?",
):
    author = event.get("author", "agent") if isinstance(event, dict) else "agent"
    content = event.get("content", {}) if isinstance(event, dict) else {}
    for part in content.get("parts", []):
        if "text" in part:
            print(f"  [{author}] {part['text'][:300]}")

## 9. Clean Up

Delete the deployed agent to avoid ongoing charges.
Uncomment and run when done testing.

In [ ]:
# Uncomment to delete the deployed agent:
# remote_agent.delete()
# print("Agent Engine resource deleted.")